# Retrieval-Augmented Generation (RAG) with LLaMA 3.2-1B on SciQ

This notebook implements a **RAG pipeline** for science question answering using the [SciQ dataset](https://huggingface.co/datasets/allenai/sciq).

**Pipeline:**
1. Load and embed context passages using `all-mpnet-base-v2`
2. Store embeddings in a FAISS vector index
3. For each question, retrieve the most relevant context
4. Generate an answer using **LLaMA 3.2-1B** conditioned on the retrieved context

**Evaluation:**
- Retrieval: Context Recall, Context Precision
- Generation: ROUGE-1, F1, Semantic Similarity

---
### Requirements
```
pip install pandas llama-index llama-index-llms-huggingface llama-index-vector-stores-faiss llama-index-embeddings-huggingface faiss-cpu sentence-transformers transformers rouge-score
```

## 1. Install Dependencies

In [ ]:
!pip install pandas llama-index llama-index-llms-huggingface llama-index-vector-stores-faiss llama-index-embeddings-huggingface faiss-cpu sentence-transformers transformers rouge-score pyarrow

## 2. Imports

In [ ]:
import pandas as pd
import faiss
import torch
import re
import os
from collections import Counter

from llama_index.core import Document, StorageContext, VectorStoreIndex
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM
from rouge_score import rouge_scorer
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 3. Configuration

In [ ]:
# Model & Paths
LLM_PATH   = "meta-llama/Llama-3.2-1B"
EMBED_PATH = "sentence-transformers/all-mpnet-base-v2"
INDEX_PATH = "faiss_index"    # saved locally, gitignored
RESULTS    = "results"

os.makedirs(RESULTS, exist_ok=True)

## 4. Load Dataset

We use the **test split** of SciQ for evaluation. Each row has a context passage, a question, and the correct answer.

In [ ]:
# Load directly from HuggingFace
dataset = load_dataset("allenai/sciq")

df = pd.DataFrame(dataset["test"])
df = df.rename(columns={
    "support":        "Context",
    "question":       "Question",
    "distractor1":    "Distractor1",
    "distractor2":    "Distractor2",
    "distractor3":    "Distractor3",
    "correct_answer": "Answer"
})[["Context", "Question", "Distractor1", "Distractor2", "Distractor3", "Answer"]]

## 5. Build Documents for Indexing

In [ ]:
# Create documents from context passages (drop empty contexts)
documents = [
    Document(text=context)
    for context in df["Context"].dropna().tolist()
    if context.strip() != ""
]

## 6. Embedding Model

In [ ]:
# Use CPU for embedding to avoid CUDA conflicts
embed_model = HuggingFaceEmbedding(
    model_name=EMBED_PATH,
    device="cpu"
)

## 7. FAISS Index

We save the index to disk after building it. On subsequent runs it loads from disk directly — no need to rebuild.

In [ ]:
dimension   = 768
faiss_index = faiss.IndexFlatL2(dimension)
vector_store    = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

if os.path.exists(INDEX_PATH):
    print("Loading existing index from disk...")
    from llama_index.core import load_index_from_storage
    storage_context = StorageContext.from_defaults(
        vector_store=vector_store,
        persist_dir=INDEX_PATH
    )
    index = load_index_from_storage(storage_context, embed_model=embed_model)
else:
    print("Building index from scratch...")
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        embed_model=embed_model
    )
    index.storage_context.persist(persist_dir=INDEX_PATH)
    print(f"Index saved to {INDEX_PATH}/")

## 8. Load LLaMA Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLM_PATH)

model = AutoModelForCausalLM.from_pretrained(
    LLM_PATH,
    device_map  = "auto",
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
)
model.eval()

## 9. RAG Pipeline

For each question:
1. **Retrieve** the most similar context from FAISS
2. **Generate** an answer using LLaMA conditioned on that context

In [ ]:
retriever = index.as_retriever(similarity_top_k=1)

def rag_generate_answer(question):
    nodes = retriever.retrieve(question)

    if len(nodes) == 0:
        return "NO_CONTEXT", None

    context = nodes[0].text

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not present in the context say "I don't know". Do not guess.
Give only the final answer.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 10,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer  = decoded.split("Answer:")[-1].strip().split("\n")[0]

    return answer, context

## 10. Run Inference

In [ ]:
print("Starting inference...")
results = []

for i, row in enumerate(df.itertuples()):
    answer, retrieved_context = rag_generate_answer(row.Question)
    results.append({
        "Question":          row.Question,
        "Retrieved_Context": retrieved_context,
        "Predicted_Answer":  answer,
    })


rag_results_df = pd.DataFrame(results)
rag_results_df.to_csv(f"{RESULTS}/rag_results.csv", index=False)
rag_results_df.head(3)

## 11. Merge with Ground Truth

In [ ]:
# Merge correct answer and original context from dataset
rag_results_df = rag_results_df.merge(
    df[["Question", "Answer", "Context"]],
    on="Question",
    how="left"
)

rag_results_df.head(3)

## 12. Normalize Text

In [ ]:
def normalize(text):
    if text is None or isinstance(text, float):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"[^a-z0-9 ]", "", text)
    return re.sub(r"\s+", " ", text).strip()

rag_results_df["pred_norm"] = rag_results_df["Predicted_Answer"].apply(normalize)
rag_results_df["gold_norm"] = rag_results_df["Answer"].apply(normalize)

## 13. Retrieval Evaluation
Calculate Recall and Precision

In [ ]:
# Context Recall
def context_recall(retrieved_context, original_context):
    if not retrieved_context or not original_context:
        return 0.0
    return int(normalize(retrieved_context) == normalize(original_context))

rag_results_df["context_recall"] = rag_results_df.apply(
    lambda x: context_recall(x["Retrieved_Context"], x["Context"]), axis=1
)
recall_score_retrieval = rag_results_df["context_recall"].mean()

# Context Precision
def context_precision(retrieved_context, question):
    if not retrieved_context or not question:
        return 0.0
    ctx_tokens = set(normalize(retrieved_context).split())
    q_tokens   = set(normalize(question).split())
    common     = ctx_tokens & q_tokens
    if len(ctx_tokens) == 0:
        return 0.0
    return len(common) / len(ctx_tokens)

rag_results_df["context_precision"] = rag_results_df.apply(
    lambda x: context_precision(x["Retrieved_Context"], x["Question"]), axis=1
)
precision_score_retrieval = rag_results_df["context_precision"].mean()

## 14. Generation Evaluation

In [ ]:
# Semantic Similarity
embedder = SentenceTransformer(EMBED_PATH, device="cpu")

pred_emb = embedder.encode(rag_results_df["Predicted_Answer"].tolist(), convert_to_tensor=True)
gold_emb = embedder.encode(rag_results_df["Answer"].tolist(),           convert_to_tensor=True)
cos_sim  = util.cos_sim(pred_emb, gold_emb).diagonal().cpu().numpy()

rag_results_df["semantic_similarity"] = cos_sim
semantic_similarity_score = rag_results_df["semantic_similarity"].mean()

In [ ]:
# ROUGE-1
scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)

def compute_rouge1(pred, gold):
    return scorer.score(gold, pred)["rouge1"].fmeasure

rag_results_df["rouge1"] = rag_results_df.apply(
    lambda x: compute_rouge1(x["pred_norm"], x["gold_norm"]), axis=1
)
rouge1_score = rag_results_df["rouge1"].mean()

In [ ]:
# F1 Score
def f1_score_gen(pred, gold):
    pred_tokens = pred.split()
    gold_tokens = gold.split()
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)
    common     = Counter(pred_tokens) & Counter(gold_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall    = num_common / len(gold_tokens)
    return 2 * (precision * recall) / (precision + recall)

rag_results_df["f1"] = rag_results_df.apply(
    lambda x: f1_score_gen(x["pred_norm"], x["gold_norm"]), axis=1
)
f1_score = rag_results_df["f1"].mean()

## 15. Final Results

In [ ]:
print("\n Retrieval Evaluation ")
print(f"Recall    : {recall_score_retrieval:.4f}")
print(f"Precision : {precision_score_retrieval:.4f}")

print("\n Generation Evaluation ")
print(f"ROUGE-1            : {rouge1_score:.4f}")
print(f"F1                 : {f1_score:.4f}")
print(f"Semantic Similarity: {semantic_similarity_score:.4f}")

# Save metrics
summary_df = pd.DataFrame([{
    "Recall_Retrieval":    recall_score_retrieval,
    "Precision_Retrieval": precision_score_retrieval,
    "ROUGE1":              rouge1_score,
    "F1":                  f1_score,
    "Semantic_Similarity": semantic_similarity_score
}])
summary_df.to_csv(f"{RESULTS}/rag_evaluation_metrics.csv", index=False)
rag_results_df.to_csv(f"{RESULTS}/rag_final_results.csv", index=False)

print(f"\nAll results saved to {RESULTS}/")
summary_df